# Evaluación cuantitativa: modelo base vs. Gemma-7b-it afinado con LoRA

Los ejemplos de `infer_gemma.ipynb` son útiles para "ver a ojo" si el
fine-tuning cambió algo, pero no tienen un resumen de referencia -así que no
se puede calcular ninguna métrica sobre ellos. Aquí usamos el split **`test`**
de `knkarthick/samsum` (resúmenes escritos por humanos, que el modelo **no**
vio durante el entrenamiento) para comparar el modelo base contra el
afinado con:

- **ROUGE-1 / ROUGE-2 / ROUGE-L / ROUGE-Lsum** — solapamiento de n-gramas
  contra la referencia (la métrica estándar para resumen automático).
- **BERTScore F1** — similitud semántica vía embeddings; a diferencia de
  ROUGE, reconoce paráfrasis ("compraron pan" vs. "fueron por pan" puntúan
  parecido aunque no comparten n-gramas).
- **Longitud promedio** del resumen generado, para ver si el modelo aprendió
  a ser tan conciso como la referencia humana (samsum pide 1-2 frases).

## Instalar dependencias adicionales


In [1]:
%pip install evaluate rouge_score bert_score absl-py pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.9 MB/s  0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24987 sha256=bde934ff7604bce636bee4f327f4569932840bcf4c085e97a17cceabdeceab2d
  Stored in directory: /home/jovyan/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [bert_score]4 [evaluate]
Note: you may need to restart the kernel to use updated packages.


## 0. Configuración

In [4]:
import os

MODEL_NAME = "google/gemma-7b-it"
ADAPTER_DIR = os.environ.get("OUTPUT_DIR", "/home/jovyan/labs/gemma-crobotp-lora")
DATA_PATH = "datos.jsonl"           # Tu archivo JSONL de 500 preguntas
EVAL_SPLIT_SIZE = 0.10              # 10% para evaluación (mismo 10% que no vio en training)
SEED = 42                           # Misma semilla para reproduibilidad
MAX_NEW_TOKENS = 256
BERTSCORE_MODEL_TYPE = "bert-base-multilingual-cased"  # Modelo multilingüe optimizado para español
OUTPUT_CSV = os.environ.get("EVAL_OUTPUT_CSV", "/home/jovyan/labs/eval_crobotp_base_vs_finetuned.csv")

if not os.path.isdir(ADAPTER_DIR) or not os.listdir(ADAPTER_DIR):
    raise SystemExit(f"No se encuentran adaptadores en {ADAPTER_DIR}. Verifica la ruta de guardado.")


## 1. Cargar el modelo base (4-bit) + los adaptadores LoRA

In [5]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("CUDA disponible:", torch.cuda.is_available())

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Modelo + adaptadores cargados.")


CUDA disponible: True


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Modelo + adaptadores cargados.


## 2. Funciones de generación (idénticas a `infer_gemma.ipynb`)

In [6]:
def responder(user_query, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{"role": "user", "content": user_query}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens, 
            do_sample=True, 
            temperature=0.2,
            top_p=0.9
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def responder_base(user_query, max_new_tokens=MAX_NEW_TOKENS):
    with model.disable_adapter():
        return responder(user_query, max_new_tokens)


## 3. Tomar una muestra del split de prueba y generar con ambos modelos

Esto es lo que toma tiempo: dos generaciones (base + afinado) por cada
ejemplo. Con `N_EXAMPLES=30` en una L4 debería tomar unos minutos; si quieres
iterar más rápido, bájalo a 10-15 primero para revisar que todo corra bien.


In [7]:
from datasets import load_dataset

# Cargar dataset local
full_dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# Crear el mismo split de evaluación usado en el entrenamiento
dataset_split = full_dataset.train_test_split(test_size=EVAL_SPLIT_SIZE, seed=SEED)
test_dataset = dataset_split["test"]

print(f"Evaluando sobre {len(test_dataset)} ejemplos del split de prueba de CROBOTP...")

prompts, references = [], []
preds_finetuned, preds_base = [], []

for i, example in enumerate(test_dataset):
    user_question = example["messages"][0]["content"]
    target_reference = example["messages"][1]["content"]
    
    prompts.append(user_question)
    references.append(target_reference)

    preds_finetuned.append(responder(user_question))
    preds_base.append(responder_base(user_question))

    if (i + 1) % 10 == 0 or (i + 1) == len(test_dataset):
        print(f"  ... {i + 1}/{len(test_dataset)} respuestas generadas")

Evaluando sobre 77 ejemplos del split de prueba de CROBOTP...
  ... 10/77 respuestas generadas
  ... 20/77 respuestas generadas
  ... 30/77 respuestas generadas
  ... 40/77 respuestas generadas
  ... 50/77 respuestas generadas
  ... 60/77 respuestas generadas
  ... 70/77 respuestas generadas
  ... 77/77 respuestas generadas


## 4. Calcular ROUGE y BERTScore

`use_aggregator=False` en la segunda llamada da el ROUGE-L **por ejemplo**
(no solo el promedio) — lo guardamos en la sección 6 para poder encontrar
los mejores/peores casos de cada modelo.


In [8]:
import evaluate

rouge = evaluate.load("rouge")

rouge_finetuned = rouge.compute(predictions=preds_finetuned, references=references)
rouge_base = rouge.compute(predictions=preds_base, references=references)

rouge_finetuned_per_example = rouge.compute(
    predictions=preds_finetuned, references=references, use_aggregator=False
)
rouge_base_per_example = rouge.compute(
    predictions=preds_base, references=references, use_aggregator=False
)

print("ROUGE (base):     ", rouge_base)
print("ROUGE (afinado):  ", rouge_finetuned)

ROUGE (base):      {'rouge1': np.float64(0.15865758355469511), 'rouge2': np.float64(0.0629822738758037), 'rougeL': np.float64(0.13801468251085347), 'rougeLsum': np.float64(0.14259602920380618)}
ROUGE (afinado):   {'rouge1': np.float64(0.3445463751239065), 'rouge2': np.float64(0.1459613098991966), 'rougeL': np.float64(0.2876926285515278), 'rougeLsum': np.float64(0.28750325020413153)}


In [9]:
bertscore = evaluate.load("bertscore")

bs_finetuned = bertscore.compute(
    predictions=preds_finetuned, references=references,
    lang="es", model_type=BERTSCORE_MODEL_TYPE,
)
bs_base = bertscore.compute(
    predictions=preds_base, references=references,
    lang="es", model_type=BERTSCORE_MODEL_TYPE,
)

bertscore_f1_base = sum(bs_base["f1"]) / len(bs_base["f1"])
bertscore_f1_finetuned = sum(bs_finetuned["f1"]) / len(bs_finetuned["f1"])

print("BERTScore F1 (base):    ", round(bertscore_f1_base, 4))
print("BERTScore F1 (afinado): ", round(bertscore_f1_finetuned, 4))

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore F1 (base):     0.654
BERTScore F1 (afinado):  0.7746


## 5. Tabla comparativa

In [10]:
import pandas as pd

df_resumen = pd.DataFrame({
    "modelo": ["base (sin LoRA)", "afinado (con LoRA)"],
    "rouge1": [rouge_base["rouge1"], rouge_finetuned["rouge1"]],
    "rouge2": [rouge_base["rouge2"], rouge_finetuned["rouge2"]],
    "rougeL": [rouge_base["rougeL"], rouge_finetuned["rougeL"]],
    "rougeLsum": [rouge_base["rougeLsum"], rouge_finetuned["rougeLsum"]],
    "bertscore_f1": [bertscore_f1_base, bertscore_f1_finetuned],
    "longitud_promedio_palabras": [
        sum(len(p.split()) for p in preds_base) / len(preds_base),
        sum(len(p.split()) for p in preds_finetuned) / len(preds_finetuned),
    ],
})

referencia_len = sum(len(r.split()) for r in references) / len(references)
print(f"(Longitud promedio de la respuesta de referencia: {referencia_len:.1f} palabras)")
df_resumen

(Longitud promedio de la respuesta de referencia: 16.4 palabras)


,modelo,rouge1,rouge2,rougeL,rougeLsum,bertscore_f1,longitud_promedio_palabras
0,base (sin LoRA),0.158658,0.062982,0.138015,0.142596,0.654021,124.506494
1,afinado (con LoRA),0.344546,0.145961,0.287693,0.287503,0.774609,16.246753


## 6. Guardar el detalle por ejemplo (para inspección manual)

Las columnas `rougeL_base` / `rougeL_afinado` te dejan ordenar y encontrar
los mejores y peores casos de cada modelo, en vez de solo mirar el promedio.


In [11]:
df_detalle = pd.DataFrame({
    "pregunta": prompts,
    "referencia": references,
    "respuesta_base": preds_base,
    "respuesta_afinada": preds_finetuned,
    "rougeL_base": rouge_base_per_example["rougeL"],
    "rougeL_afinado": rouge_finetuned_per_example["rougeL"],
})

df_detalle.to_csv(OUTPUT_CSV, index=False)
print(f"Detalle de respuestas guardado en: {OUTPUT_CSV}")

# Los 3 ejemplos con mayor mejora en ROUGE-L tras el fine-tuning
df_detalle["mejora"] = df_detalle["rougeL_afinado"] - df_detalle["rougeL_base"]
df_detalle.sort_values("mejora", ascending=False).head(3)[
    ["pregunta", "referencia", "respuesta_base", "respuesta_afinada", "mejora"]
]

Detalle de respuestas guardado en: /home/jovyan/labs/eval_crobotp_base_vs_finetuned.csv


,pregunta,referencia,respuesta_base,respuesta_afinada,mejora
6,¿Qué comando marca la conclusión de la tabla d...,El comando END.,"La respuesta es: ""END"" (Final). El comando ""EN...",El comando END.,0.760000
64,¿Cuál es el alcance máximo (Maximum reaching d...,Su alcance máximo es de 916 mm.,El alcance máximo (Maximum reaching distance) ...,Su alcance máximo es de 915 mm.,0.504630
49,¿Cuál es el peso propio del robot CRP-RA09A-07?,El peso total del cuerpo es de 46 kg.,El peso propio del robot CRP-RA09A-07 no se in...,El peso propio del robot es de 142 kg.,0.504505


## Notas finales

- **¿Por qué comparar contra el split `test` y no `train`?** Porque evaluar
  sobre ejemplos que el modelo ya vio durante el entrenamiento sobreestima
  qué tan bien generaliza -el punto de un split de prueba es medir
  desempeño en datos nuevos.
- **ROUGE vs. BERTScore:** si el modelo afinado mejora en ROUGE pero no en
  BERTScore (o viceversa), probablemente esté aprendiendo el *formato*
  (longitud, estilo telegráfico de samsum) más que el *contenido semántico*
  -vale la pena mirar `df_detalle` para casos concretos antes de concluir algo.
- **Extensión opcional (LLM-as-judge):** además de ROUGE/BERTScore, una
  práctica cada vez más común en 2026 es pedirle a un modelo más fuerte
  (Claude, GPT, Gemini) que califique cada resumen en una escala de
  fidelidad/coherencia frente al diálogo original -es más caro y más lento,
  pero puede detectar errores (alucinaciones, mezclar quién dijo qué) que
  ROUGE/BERTScore no ven porque solo miran similitud con la referencia.
- Este mismo pipeline existe como script plano en `evaluate_gemma.py`
  (con flags `--n_examples`, `--skip_bertscore`, etc.) para correrlo de una
  sola vez sin pasar por el notebook.
